# 02 - Dataset Preparation

One supervised fine-tuning set per expert:

| domain | source | why |
|---|---|---|
| legal | LegalBench (`nguha/legalbench`) | already cited in the paper's related work |
| news | NewsQA (`lucadiliello/newsqa`) | preprocessed mirror of Trischler et al. 2017; the official `Maluuba/newsqa` needs a manual Microsoft download |
| general | Dolly-15k (`databricks/databricks-dolly-15k`) | human-written, so no training on another model's outputs |

**Two defects in the naive version of this, both of which produce adapters that
look fine by loss and are useless in practice.**

## 2.1 Defect 1 - benchmark targets are far too short

LegalBench and NewsQA are *evaluation* benchmarks. Measured over our sample:

- **105 of 112** loadable LegalBench tasks have mean answers of **10 characters
  or fewer**; the median answer is **3 characters** (`Yes`, `No`, `UCC`).
- Only **42 of 2375** sampled rows exceed 40 characters.
- NewsQA answers are extractive spans averaging **26 characters**.

The deployed agents are prompted for structured multi-sentence answers and
scored against long references. Training directly on those targets teaches an
expert to answer in three characters, which the judge scores as incomplete -
an artefact that presents as *'specialisation made quality worse'*.

In [ ]:
# Reproduce the measurement.
import json, collections, statistics
rows = [json.loads(l) for l in open('finetune/data/legal_train.jsonl', encoding='utf-8')]
by_task = collections.defaultdict(list)
for r in rows:
    by_task[r['source']].append(len(r.get('raw_answer', '')))
means = sorted(((statistics.mean(v), k) for k, v in by_task.items()), reverse=True)
print('tasks:', len(means))
print('with mean raw answer <= 10 chars:', sum(1 for m, _ in means if m <= 10))
for m, k in means[:5]:
    print(f'  {m:7.1f} chars  {k}')

**Fix:** render each target into the response format the agent is served
behind, using only content already in the dataset - the task's question, the
facts it supplies, and its ground-truth determination.

Fields the datasets do not provide (notably **statutory citations**) are
**omitted rather than filled**. Training a legal assistant to produce
plausible-looking citations it cannot support is the worst available failure
mode. No model generates any part of any training target.

Result: mean target length **3.8 -> 420 chars** (legal), **26 -> 216** (news).

## 2.2 Defect 2 - examples longer than the training window lose their target

All three datasets put the target at the **end** of the rendered example, so an
example longer than `--max-seq-length` is right-truncated into an input with
**no answer attached**. A model trained on those produces a flat,
plausible-looking loss curve rather than an error.

At a 1400-char input cap and 512-token training window, **83% of NewsQA
examples were over the limit** (NewsQA embeds a full article ahead of the
question). The resulting news adapter did not learn: held-out loss ran
**1.920 -> 1.923 -> 1.956** across three epochs with token accuracy static at
0.568, while legal over the same budget improved 0.919 -> 0.844.

The rate was also badly **uneven** - 83% news / 22% legal / 9% general - which
is itself an inconsistency in effective training budget.

In [ ]:
# Dataset preparation. Defaults encode the fixes:
#   MAX_INPUT_CHARS = 2800        (sized against a 1024-token training window)
#   --legal-format irac          (structured targets, not bare labels)
#   --news-format grounded       (answer span + the article sentence containing it)
#   NewsQA articles are WINDOWED around the answer, not truncated from the start,
#   because head-truncation often removes the very sentence the answer came from.
!.venv\Scripts\python.exe finetune\prepare_datasets.py --domain all

`prepare_datasets.py` now **tokenises the rendered examples with each model's
own tokenizer** and refuses to pass quietly above a 10% over-limit rate. That
check is what makes this class of failure impossible to ship silently again.

Final rates: **legal 0.0% / news 0.0% / general 0.7%**.

In [ ]:
# Inspect the manifest - this is the record to check BEFORE training.
import json
m = json.load(open('finetune/data/manifest.json', encoding='utf-8'))
for domain, info in m['domains'].items():
    tok = info['token_lengths']
    print(f"{domain:8s} n={info['train_examples']:5d} "
          f"mean_target={info['mean_output_chars']:7.1f} chars  "
          f"median={tok['median_tokens']:4d} tok  over_limit={tok['pct_over_limit']}%")
print()
print('cross-domain overlap:', m['cross_domain_overlap'])
print('target-length check :', m['target_length_check']['domains'])

In [ ]:
# Eyeball actual training examples. Do this - the numbers above cannot tell you
# whether the target text is sensible.
import json
for domain in ('legal', 'news', 'general'):
    r = json.loads(open(f'finetune/data/{domain}_train.jsonl', encoding='utf-8').readline())
    print('=' * 70)
    print(f"{domain}  [{r['source']}]")
    print('INSTRUCTION:', r['instruction'][:160])
    print('INPUT      :', r['input'][:200].replace(chr(10), ' | '))
    print('TARGET     :', r['output'][:300].replace(chr(10), ' | '))

## 2.3 Cross-domain leakage control

The pivot plan asks that no query or source text be shared between the legal and
news sets. Enforced two ways, both recorded in the manifest: exact dedupe by
normalised hash, and token-Jaccard >= 0.6 against any legal record sharing a
rare token (an inverted index keeps this from being an all-pairs comparison).

On this sample the filter removed **nothing**, which is the expected outcome for
contract-law tasks against CNN news articles. Reported rather than presented as
a validated safeguard.